In [ ]:
import subprocess
from pathlib import Path
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, SDWriter
from rdkit.Chem.MolStandardize import rdMolStandardize
from dimorphite_dl import protonate_smiles

lfc = rdMolStandardize.LargestFragmentChooser()

COHORTS = Path("Ligands/cohorts")
CHEMOTYPES = ["K1", "P1", "P2", "T1", "S1"]
NCONF = 5

In [ ]:
def load_cohort(ct):
    tight = COHORTS / ct / f"{ct}_cohort_tight.csv"
    full  = COHORTS / ct / f"{ct}_cohort.csv"
    path = tight if tight.exists() else full
    df = pd.read_csv(path)
    print(f"{ct}: {len(df)} from {path.name}")
    return df

def safe_name(name):
    return "".join(c if c.isalnum() or c in "-_" else "_" for c in str(name))[:50]

def already_prepped(out_dir, name):
    hits = list(out_dir.glob(f"{name}.sdf")) + list(out_dir.glob(f"{name}_p*.sdf"))
    return any(p.exists() and p.stat().st_size > 0 for p in hits)

### Ligand preparation workflow: protonation (pH 7.0-7.8), fragment cleanup, conformer embedding, MMFF optimisation, SDF export

In [ ]:
# See full Lig_Prep.ipynb in Drive Onco/repo for complete prep_one() and batch loops.
# Core stack: RDKit + dimorphite_dl + MMFF conformers → Ligands/{chemotype}/prepped/*.sdf